In [8]:
# ── Imports standard ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
import os

os.chdir("/home/onyxia/work/PESSD") 


# ══════════════════════════════════════════════════════════════════════════════
# Charger les données
# ══════════════════════════════════════════════════════════════════════════════

total_deces = pd.read_excel('data/total_deces.xlsx', header=0)
pop_totale  = pd.read_excel('data/Pop_tot_fr.xlsx', header=0)

print('✅ Fichiers chargés')
print(f'   Total décès : {total_deces.shape}')
print(f'   Pop totale  : {pop_totale.shape}')

# ══════════════════════════════════════════════════════════════════════════════
# Conversion en format long
# ══════════════════════════════════════════════════════════════════════════════

# Décès
deces_long = total_deces.rename(columns={'TIME': 'Pays'}).melt(
    id_vars='Pays', var_name='Annee', value_name='Deces'
)
deces_long['Annee'] = deces_long['Annee'].astype(int)
deces_long['Deces'] = pd.to_numeric(
    deces_long['Deces'].astype(str).str.replace(':', 'nan').str.strip(),
    errors='coerce'
)

# Population
pop_long = pop_totale.rename(columns={'TIME': 'Pays'}).melt(
    id_vars='Pays', var_name='Annee', value_name='Pop_totale'
)
pop_long['Annee'] = pop_long['Annee'].astype(int)
pop_long['Pop_totale'] = pd.to_numeric(
    pop_long['Pop_totale'].astype(str).str.replace(':', 'nan').str.strip(),
    errors='coerce'
)

print('✅ Conversion en format long')

# ══════════════════════════════════════════════════════════════════════════════
# Calcul : (Décès t + Décès t+1) / Pop totale t
# ══════════════════════════════════════════════════════════════════════════════

ANNEE_DEBUT = 2004
ANNEE_FIN   = 2023

PAYS_LIST = deces_long['Pays'].unique()
all_results = []

for pays in PAYS_LIST:
    # Décès du pays
    d = deces_long[
        (deces_long['Pays'] == pays) &
        (deces_long['Annee'] >= ANNEE_DEBUT) &
        (deces_long['Annee'] <= ANNEE_FIN + 1)
    ]
    
    # Pop totale du pays
    p = pop_long[
        (pop_long['Pays'] == pays) &
        (pop_long['Annee'] >= ANNEE_DEBUT) &
        (pop_long['Annee'] <= ANNEE_FIN)
    ]
    
    # Décès année t
    D_t = d.set_index('Annee')['Deces']
    
    # Décès année t+1 (shift -1 pour ramener sur indice t)
    D_t1 = d.set_index('Annee')['Deces'].shift(-1)
    
    # Assemblage
    df_r = pd.DataFrame({
        'D_t': D_t,
        'D_t1': D_t1,
        'Pop_totale': p.set_index('Annee')['Pop_totale']
    }).loc[ANNEE_DEBUT:ANNEE_FIN]
    
    # Taux = (Décès t + Décès t+1) / Pop totale t × 100
    df_r['Taux_deces_2ans'] = (df_r['D_t'] + df_r['D_t1']) / df_r['Pop_totale'] * 100
    df_r['Pays'] = pays
    df_r.index.name = 'Annee'
    all_results.append(df_r.reset_index())

df_all = pd.concat(all_results, ignore_index=True)
valid  = df_all.dropna(subset=['Taux_deces_2ans'])

print(f'\n✅ Panel construit : {len(valid):,} obs. valides')
print(f'   {valid.Pays.nunique()} pays × {valid.Annee.nunique()} années ({valid.Annee.min()}–{valid.Annee.max()})')

# ══════════════════════════════════════════════════════════════════════════════
# Export pivot (Pays × Années)
# ══════════════════════════════════════════════════════════════════════════════

df_pivot = valid.pivot(index='Pays', columns='Annee', values='Taux_deces_2ans').round(3)
df_pivot.columns.name = None
df_pivot.index.name   = 'Pays'

os.makedirs('data', exist_ok=True)
df_pivot.to_excel('data/Taux_Deces_2ans.xlsx')

print(f'\n✅ Export : data/Taux_Deces_2ans.xlsx')
print(f'   {df_pivot.shape[0]} pays × {df_pivot.shape[1]} années')

print('\nAperçu (5 premiers pays) :')
print(df_pivot.head())


✅ Fichiers chargés
   Total décès : (21, 22)
   Pop totale  : (21, 34)
✅ Conversion en format long

✅ Panel construit : 402 obs. valides
   21 pays × 20 années (2004–2023)

✅ Export : data/Taux_Deces_2ans.xlsx
   21 pays × 20 années

Aperçu (5 premiers pays) :
            2004   2005   2006   2007   2008   2009   2010   2011   2012  \
Pays                                                                       
Allemagne  1.997  2.002  2.000  2.031  2.066  2.089  2.092  2.146  2.195   
Autriche   1.836  1.823  1.804  1.807  1.835  1.855  1.840  1.862  1.891   
Belgique   1.967  1.961  1.924  1.939  1.954  1.944  1.932  1.940  1.972   
Danemark   2.052  2.041  2.047  2.023  1.999  1.982  1.931  1.885  1.878   
Espagne    1.774  1.743  1.710  1.714  1.680  1.651  1.648  1.687  1.687   

            2013   2014   2015   2016   2017   2018   2019   2020   2021  \
Pays                                                                       
Allemagne  2.188  2.221  2.261  2.243  2.287  2.288  2

In [7]:
PATH_DECES = 'data/total_deces.xlsx'   
PATH_POP   = 'data/Pop_tot_fr.xlsx'
PATH_OUTPUT = 'data/Taux_Deces_2ans.xlsx'

def charger_fichier(path, label):
    if path is None or not os.path.exists(path):
        if IN_COLAB:
            print(f'📂 Upload : {label}')
            uploaded = files.upload()
            return pd.read_excel(list(uploaded.keys())[0], header=0)
        raise FileNotFoundError(f'Introuvable : {path}')
    return pd.read_excel(path, header=0)

df_deces_raw = charger_fichier(PATH_DECES, 'total_deces.xlsx')
df_pop_raw   = charger_fichier(PATH_POP,   'Pop_tot_fr.xlsx')
